# Benchmark — Fase 3: análisis y gráficas

Lee los CSV que producen las Fases 1 y 2 y los grafica para decidir con la vista:

- `outputs/benchmark/detectors.csv` → eficiencia de detectores (Fase 1).
- `outputs/benchmark/trackers.csv` → eficiencia + consistencia de los 4 trackers (Fase 2).

**Sin ground-truth:** todo lo de aquí mide **eficiencia y consistencia**, NO exactitud
vs. humano. Las gráficas anotan, para cada métrica, si **mayor o menor es mejor**.

> Si una de las dos fases aún no se corrió, su sección se salta sola (el CSV no existe).

## Paso 0 — Cargar CSVs y utilidades de gráficas

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src.utils import PROJECT_ROOT

BENCH = PROJECT_ROOT / "outputs" / "benchmark"


def load_csv(name):
    p = BENCH / name
    if not p.exists():
        print(f"[falta] {p}  (corre la fase correspondiente primero)")
        return None
    df = pd.read_csv(p)
    print(f"[ok] {name}: {len(df)} filas")
    return df


det_df = load_csv("detectors.csv")
trk_df = load_csv("trackers.csv")


def bars(ax, df, col, better, color="#4C72B0"):
    # Barra por config para una metrica, con anotacion de valor y de 'mejor cuando'.
    s = df.set_index("config")[col].dropna()
    if s.empty:
        ax.set_title(f"{col} (sin datos)")
        ax.axis("off")
        return
    ax.bar(range(len(s)), s.values, color=color)
    ax.set_xticks(range(len(s)))
    ax.set_xticklabels(s.index, rotation=25, ha="right", fontsize=8)
    ax.set_title(f"{col}  ({better})", fontsize=10)
    vmax = max(s.values)
    for i, v in enumerate(s.values):
        ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=7)
    ax.set_ylim(0, vmax * 1.18 if vmax > 0 else 1)

## Fase 1 — eficiencia de detectores

`fps` (↑ mejor) y `peak_vram_mb` (↓ mejor). Es la única comparación rigurosa sin GT:
elige el detector más eficiente para llevarlo a la Fase 2.

In [ ]:
if det_df is not None:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    bars(axes[0], det_df, "fps", "↑ mejor", color="#55A868")
    bars(axes[1], det_df, "peak_vram_mb", "↓ mejor", color="#C44E52")
    fig.suptitle("Fase 1 — eficiencia por detector", fontsize=12)
    fig.tight_layout()
    plt.show()
else:
    print("Fase 1 sin datos: corre 01_benchmark_detectors.ipynb")

## Fase 2 — eficiencia y consistencia (2×2)

Cuatro configs (detector × tracker). **Robustas:** `fps`/`peak_vram_mb` (eficiencia) y
`frag_rate` (↓) / `tracklet_len` (↑) (consistencia del tracking).

In [ ]:
if trk_df is not None:
    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    bars(axes[0][0], trk_df, "fps", "↑ mejor", color="#55A868")
    bars(axes[0][1], trk_df, "peak_vram_mb", "↓ mejor", color="#C44E52")
    bars(axes[1][0], trk_df, "frag_rate", "↓ mejor", color="#4C72B0")
    bars(axes[1][1], trk_df, "tracklet_len", "↑ mejor", color="#8172B3")
    fig.suptitle("Fase 2 — 2×2: eficiencia (arriba) y consistencia (abajo)", fontsize=12)
    fig.tight_layout()
    plt.show()
else:
    print("Fase 2 sin datos: corre 02_benchmark_trackers.ipynb")

## Fase 2 — aislar cada eje

Como el detector domina el costo y el tracker domina la consistencia, conviene mirar
cada eje por separado: **eficiencia promediada por detector** y **consistencia
promediada por tracker**. Esto separa "qué detector" de "qué tracker".

In [ ]:
if trk_df is not None:
    d = trk_df.copy()
    # label "detector+tracker" -> dos columnas (el detector no contiene '+').
    d[["detector", "tracker"]] = d["config"].str.split("+", n=1, expand=True)

    by_det = d.groupby("detector", as_index=True)[["fps", "peak_vram_mb"]].mean()
    by_trk = d.groupby("tracker", as_index=True)[["frag_rate", "tracklet_len"]].mean()

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    by_det.plot.bar(ax=axes[0], rot=0)
    axes[0].set_title("Eficiencia promedio POR DETECTOR\n(fps ↑, vram ↓)", fontsize=10)
    axes[0].set_xlabel("")
    by_trk.plot.bar(ax=axes[1], rot=0)
    axes[1].set_title("Consistencia promedio POR TRACKER\n(frag_rate ↓, tracklet_len ↑)", fontsize=10)
    axes[1].set_xlabel("")
    fig.tight_layout()
    plt.show()

    print("Eficiencia por detector:\n", by_det.round(2), "\n")
    print("Consistencia por tracker:\n", by_trk.round(2))
else:
    print("Fase 2 sin datos.")

## Métricas débiles (con reserva)

`smoothness` (↓), `mask_iou` (↑) y `com_jitter` (↓) están **confundidas** por el
movimiento real del objeto, diluidas por clases estáticas (`green_floor`) y son
*gameables*. Se grafican por completitud, **no para decidir**.

In [ ]:
if trk_df is not None:
    weak_cols = [("smoothness", "↓ mejor"), ("mask_iou", "↑ mejor"), ("com_jitter", "↓ mejor")]
    present = [(c, b) for c, b in weak_cols if c in trk_df and trk_df[c].notna().any()]
    if present:
        fig, axes = plt.subplots(1, len(present), figsize=(4 * len(present), 4))
        if len(present) == 1:
            axes = [axes]
        for ax, (c, b) in zip(axes, present):
            bars(ax, trk_df, c, b, color="#937860")
        fig.suptitle("Fase 2 — métricas débiles (suplementarias)", fontsize=12)
        fig.tight_layout()
        plt.show()
    else:
        print("Sin métricas de máscara (¿corriste la Fase 2 con INCLUDE_MASKS=True?)")
else:
    print("Fase 2 sin datos.")

## Cómo concluir

1. **Detector:** de la Fase 1, el de mayor `fps` y menor `peak_vram_mb`. (Confirmado por
   "eficiencia promedio por detector" de la Fase 2.)
2. **Tracker:** de la Fase 2, el de menor `frag_rate` y mayor `tracklet_len`
   ("consistencia promedio por tracker"). La eficiencia entre trackers es casi igual.
3. **Interacción:** revisa el 2×2 por si la mejor pareja no es la combinación ingenua
   de los dos ganadores por separado.

La **decisión final es humana**: estas gráficas informan el balance
eficiencia↔consistencia. La **exactitud** (mIoU/mAP/MOTA) solo llegará con la evaluación
contra ground-truth (trabajo futuro, anotación del equipo en Roboflow).